# 딥러닝 - 다중선형회귀 + 하이퍼 파라미터 튜닝

## 1. 준비작업
### 1. 패키지 가져오기

In [ ]:
# 원래 3.11.9
from hossam import *

from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np

from datetime import datetime as t
from keras_tuner import Hyperband

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import RootMeanSquaredError, R2Score

from tqdm.keras import TqdmCallback

📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.5.4
현재 설치된 'hossam' 패키지 버전: 0.5.4


: 

### 2. 데이터셋 준비하기

In [ ]:
origin = load_data('fish')
print(f"데이터셋 크기: {origin.shape}")
print(f"열 개수: {origin.shape[1]}")
print(f"행 개수: {origin.shape[0]}")
print(origin.info())
origin.head()

### 3. 랜덤시드 고정

In [ ]:
# tensorflow는 numpy를 통해 전역적으로 랜덤시드를 고정
np.random.seed(52)

## 2. 탐색적 데이터 분석
### 1. 데이터 품질 검사

In [ ]:
desc = origin.describe().T

# 숫자형 컬럼 이름만 추출
num_cols = origin.select_dtypes(include = np.number).columns

# 왜도 확인 및 로그 변환 필요성
for column in num_cols:
    skewness = origin[column].skew()
    if abs(skewness) < 0.5:
        strength = 'week'
        log_transform = 'not needed'
    elif abs(skewness) < 1:
        strength = 'normal'
        log_transform = 'recommended'
    else:
        strength = 'strong'
        log_transform = 'needed'

    desc.loc[column, 'skewness'] = skewness
    desc.loc[column, 'skewness_strength'] = strength
    desc.loc[column, 'log_transform'] = log_transform
desc

### 2. 무게에 대한 로그 변환


In [ ]:
df = origin.copy()
df['무게'] = np.log1p(df['무게'])
df.head()

## 3. 데이터 전처리
### 1. 훈련 / 검증 데이터 분리

In [ ]:
yname = '무게'
x = df.drop(columns = [yname])
y = df[yname]

x_train, y_train, x_test, y_test = train_test_split(x, y, test_size = 0.25, random_state = 52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

# 원래는 다중 공선성 제거도 고려해봐야 한다
_, cols = x_train.shape

## 4. 신경망 모델 적합
### 1. 하이퍼파라미터 튜닝 정의 - 튜닝할 파라미터 설정하는 콜백함수를 정의해야 함

In [ ]:
def tf_build(hp) -> Sequential:
    model = Sequential()

    # 입력층 정의
    model.add(Input(shape = (cols,)))

    # 은닉층 --> 유닛 수를 하이퍼파라미터로 조정
    model.add(
        Dense(
            units = hp.Choice('units', values = [4, 8, 16, 32, 64]),
            activation = 'relu'
        )
    )

    # 출력층: 1개의 뉴런 --> 하나의 값을 출력
    # 선형회귀 모델이기 때문에 활성화 함수는 linear
    model.add(Dense(1, activation = 'linear'))

    # 모델 학습 설정 (컴파일 단계)
    model.compile(
        optimizer = 'adam',
        loss = 'mse',
        # mae: 평균절대오차, rmse: 평균제곱근오차, R^2: 결정계수
        metrics = ['mae', RootMeanSquaredError(name = 'rmse'), R2Score(name = 'r2')]
    )

    return model

### 2. 튜너 객체 생성
- RandomSearch 혹은 Hyperband 튜너를 인스턴스화하여 하이퍼튜닝을 수행한다
- Hyperband 튜너를 인스턴스화하려면 최적화할 하이퍼모델인 objective 및 훈련할 최대 epoch 수(max_epochs)를 지정해야 한다
- Hyperband 튜닝 알고리즘은 적응형 리소스 할당 및 조기 중단을 사용하여 고성능 모델에서 신속하게 수렴한다.
- 이 알고리즘은 몇 개의 epoch에 대해 많은 수의 모델을 훈련하고 최고 성능을 보이는 절반만 다음 단계로 넘긴다.


In [ ]:
tuner = Hyperband(
    hypermodel = tf_build,       # 하이퍼파라미터를 튜닝하기 위한 모델 생성 함수
    objective = 'val_mae',       # 최적화 기준값
    max_epochs = 10,
    factor = 3,
    seed = 52,
    directory = 'D:\\tensor_hyperband',
    project_name = 'tf_hyperband_%s' % dt.now().strftime('%Y%m%d%H%M%S')
)

tuner

### 3. 하이퍼파라미터 튜닝 수행

In [ ]:
tuner.serach(
    x_train, y_train, epochs = 10, batch_size = 32, validation_data = (x_test, y_test)
)

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters()

if not best_hps:
    raise ValueError('No best hyperparameters found')

print(f'''best hyperparameters: {best_hps[0].values}''')

### 4. 최종 모형 도출

In [ ]:
model = tuner.hypermodel.build(best_hps[0])

result = model.fit(
    x_train,                                # 학습 데이터 입력
    y_train,                                # 학습 데이터의 타겟값 (정답)
    epochs = 500,                           # 전체 학습 데이터셋을 500회 반복하여 학습
    validation_data = (x_test, y_test),     # 검증 데이터 설정
    verbose = 0,                            # 학습 과정 출력 생략
    callbacks = [
        TqdmCallback(verbose = 1),          # 학습 진행 상황을 tqdm 형태로 보여주는 콜백 추가
        # 조기 종료 콜백: 검증 손실(val_loss)이 5회 연속 개선되지 않으면 학습 중단, 개선 기준은 최소 0.001 이상
        # 너무 일찍 조기종료하면 모델이 충분히 학습되지 않고 에러율이 높게 나올 수 있음
        EarlyStopping(monitor = 'val_loss', patience = 5, min_delta = 0.001),

        # 학습률 감소 콜백: 검증 손실(val_loss)이 10회 연속 개선되지 않으면 학습률을 0.5배로 감소
        ReduceLROnPlateau(monitor = 'val_loss', factor = 0.5, patience = 10, min_lr = 0, verbose = 1)
    ]
)
result

## 5. 성능평가
### 1. 성능평가 지표

In [ ]:
# Train 성능평가
# metrics = ['accuracy']에 의해 evaluate()가 반환하는 리스트 순서는 [loss, accuracy]
train_eval = model.evaluate(x_train, y_train, verbose = 0, return_dict = True)
test_eval = model.evaluate(x_test, y_test, verbose = 0, return_dict = True)

# DataFrame 생성
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, 'Dataset', ['Train', 'Test'])

# Loss gap 추가
final_results['RMSE_gap'] = None
final_results.loc[1, 'RMSE_gap'] = final_results.loc[1, 'rmse'] - final_results.loc[0, 'rmse']

final_results

### 2. 학습 과정 확인

In [ ]:
history_df = DataFrame(data = result.history)
history_df['epoch'] = history_df.index + 1

history_df.head()

### 3. Loss, RMSE 학습곡선

In [ ]:
figsize = (1600/ 100, 600 / 100)
fig, ax = plt.subplots(1, 2, figsize = figsize, dpi = 100)
fig.subplots_adjust(wspace = 0.2, hspace = 0.2)

# 훈련 데이터 손실률
sb.lineplot(data = history_df, x= 'epoch', y = 'loss', ax = ax[0], label = 'Train Loss'
           )

# 검증 데이터 손실률
sb.lineplot(data = history_df, x= 'epoch', y = 'val_loss', ax = ax[0], label = 'Validation Loss'
           )

# 그래프 꾸미기
ax.grid(True) # 배경 격자 표시/숨김
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].set_title('Training vs Validation Loss')  

ax[0].grid(True, alpha = 0.3)

# 훈련 데이터 손실률
sb.lineplot(data = history_df, x= 'epoch', y = 'rmse', ax = ax[1], label = 'Train RMSE'
           )

# 검증 데이터 손실률
sb.lineplot(data = history_df, x= 'epoch', y = 'val_rmse', ax = ax[1], label = 'Validation RMSE'
           )

# 그래프 꾸미기
ax.grid(True) # 배경 격자 표시/숨김
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('RMSE')
ax[1].set_title('Training vs Validation RMSE')  

ax[1].grid(True, alpha = 0.3)

# 출력
plt.grid()
plt.tight_layout()
plt.show()
plt.close()

## 6. 예측 결과 활용
### 1. 예측치 구하기

In [ ]:
pred = model.predict(x_test, verbose = 0)
pred[:5]

### 2. 결과 데이터 셋 구성


In [ ]:
kdf = DataFrame({
    '길이': x_test['길이'],
    '실제값': y_test,
    '예측값': pred.flattern()
})

kdf['오차'] = kdf['실제값'] - kdf['예측값']

kdf.head()

### 3. 관측치와 예측치 비교 시각화

In [ ]:
figsize = (1200 / 100, 700 / 100)
fig, ax = plt.subplots(1, 1, figsize = figsize, dpi = 100)
sb.regplot(data = kdf, x = '길이', y = '실제값', label = '실제값')
sb.regplot(data = kdf, x = '길이', y = '예측값', label = '예측값')
ax.legend()
ax.grid(True, alpha = 0.3)
ax.set_title(f'실제값 vs 예측값')
plt.tight_layout()
plt.show()
plt.close()

# 딥러닝 - 다중선형회귀 + 결과해석

## 1. 준비작업
### 1. 패키지참조

In [1]:
from hossam import *

from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np

from datetime import datetime as t
from keras_tuner import Hyoerband

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import RootMeanSquaredError, R2Score

from tqdm.keras import TqdmCallback

# VIF 계산 위한 함수
from statsmodel.stats.outliers_influence import variance_finlation_factor
import statsmodels.api as sm
import shap

📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.5.3

⚠️  'hossam' 패키지의 최신 버전이 출시되었습니다! (설치된 버전: 0.5.3, 최신 버전: 0.5.4)
   최신 버전으로 업데이트하려면 다음 명령어를 실행하세요:
   pip install --upgrade hossam



Warning: hossam 패키지가 최신 버전이 아닙니다.

## 2. 데이터셋 가져오기

In [ ]:
origin = load_data('')
print(f"데이터셋 크기: {origin.shape}")
print(f"열 개수: {origin.shape[1]}")
print(f"행 개수: {origin.shape[0]}")
print(origin.info())
origin.head()